In [5]:
import torch
torch.cuda.empty_cache()

--Install Dependency--

In [1]:
%pip install torch

--Cek Runtime Yang digunakan--

In [2]:
import torch, sys
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device count:", torch.cuda.device_count())
    print("device name:", torch.cuda.get_device_name(0))
else:
    print("No CUDA device visible")

python: c:\Users\Derta__\AppData\Local\Programs\Python\Python311\python.exe
torch: 2.9.1+cpu
cuda available: False
No CUDA device visible


--Mount Ke G-Drive--

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


--Check Library Ultralytics Di Drive--

In [3]:
# Unzip folder ultralytics dari Drive
import os
import zipfile

#zip /content/drive/MyDrive/ultralytics.zip
zip_path = "/content/drive/MyDrive/YOLOv12/ultralytics.zip"  # <-- ubah sesuai lokasi zip Anda

# Cek apakah file zip ada
if os.path.exists(zip_path):
    print(f"✓ File zip ditemukan: {zip_path}")
    
    # Unzip ke /content
    extract_to = "/content"
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    print(f"✓ Unzip selesai ke: {extract_to}")
    
    # Verifikasi folder ada
    if os.path.exists("/content/ultralytics"):
        print("✓ Folder ultralytics berhasil diunzip")
        print("Isi folder ultralytics:")
        for item in os.listdir("/content/ultralytics")[:10]:  # tampilkan 10 item pertama
            print(f"  - {item}")
    else:
        print("✗ Folder ultralytics belum ada setelah unzip")
else:
    print(f"✗ File zip tidak ditemukan di: {zip_path}")
    print("Pastikan path zip sudah benar")

✓ File zip ditemukan: /content/drive/MyDrive/YOLOv12/ultralytics.zip
✓ Unzip selesai ke: /content
✓ Folder ultralytics berhasil diunzip
Isi folder ultralytics:
  - assets
  - optim
  - py.typed
  - solutions
  - nn
  - trackers
  - __init__.py
  - engine
  - runs
  - cfg


--Install Editor Ultralytics Agar Bisa Dipakai--

In [4]:
# Install ultralytics dalam mode editable
import subprocess
import os

os.chdir("/content")

# Install paket editable agar Python pakai versi lokal yang diedit
result = subprocess.run(
    ["pip", "install", "-e", "./ultralytics"],
    capture_output=True,
    text=True
)

print("Install ultralytics editable mode:")
print(result.stdout)
if result.stderr:
    print("Warnings/Errors:", result.stderr)

# Verifikasi import
try:
    from ultralytics import YOLO
    print("✓ YOLO berhasil diimport dari ultralytics lokal")
except ImportError as e:
    print(f"✗ Error import YOLO: {e}")

Install ultralytics editable mode:
Obtaining file:///content/ultralytics

Warnings/Errors: ERROR: file:///content/ultralytics does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

✓ YOLO berhasil diimport dari ultralytics lokal


Cek File Txt

In [ ]:
import os

LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_derta_v2/data/train/labels"

corrupt_files = []

for file in os.listdir(LABEL_DIR):

    if not file.endswith(".txt"):
        continue

    path = os.path.join(LABEL_DIR, file)

    with open(path, "r") as f:
        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        # harus 5 nilai
        if len(parts) != 5:
            corrupt_files.append((file, "Jumlah kolom bukan 5"))
            continue

        try:
            cls, x, y, w, h = map(float, parts)

            # class harus 0-6
            if int(cls) not in range(7):
                corrupt_files.append((file, f"Class invalid: {cls}"))

            # koordinat harus 0-1
            for val in [x, y, w, h]:
                if val < 0 or val > 1:
                    corrupt_files.append((file, f"Koordinat invalid: {val}"))

        except:
            corrupt_files.append((file, "Format tidak bisa dibaca"))

print(f"Total corrupt labels: {len(corrupt_files)}")

for item in corrupt_files[:20]:
    print(item)

**Unzip Dataset**

In [6]:
# 📦 UNZIP & SETUP DATASET BARU - Extract dataset zip dan config otomatis
import os
import zipfile
import yaml
import shutil

print("=" * 70)
print("📦 UNZIP & SETUP DATASET BARU")
print("=" * 70)

# ⭐ UBAH PATH INI SESUAI LOKASI ZIP FILE ANDA DI DRIVE
DATASET_ZIP_PATH = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta.zip"  # <-- UBAH SESUAI LOKASI
EXTRACT_TO_PATH = "/content/drive/MyDrive/Dataset_Terbaru"  # Extract ke folder MyDrive
DATASET_NAME = "data_coffe_augmentasi_derta"  # Nama folder hasil extract

print(f"\n1️⃣ Cek file zip:")
if os.path.exists(DATASET_ZIP_PATH):
    print(f"   ✓ File zip ditemukan: {DATASET_ZIP_PATH}")
    
    # Informasi size
    file_size_mb = os.path.getsize(DATASET_ZIP_PATH) / (1024 * 1024)
    print(f"   📊 Ukuran: {file_size_mb:.2f} MB")
    
    print(f"\n2️⃣ Unzip dataset...")
    try:
        # Buat folder tujuan extract
        dataset_extract_path = os.path.join(
            EXTRACT_TO_PATH,
            DATASET_NAME,
            "data"
        )
        os.makedirs(dataset_extract_path, exist_ok=True)

        # Extract zip
        with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(dataset_extract_path)

        print(f"   ✓ Unzip selesai ke: {dataset_extract_path}")
    except Exception as e:
        print(f"   ❌ Error unzip: {e}")
        print(f"   Pastikan file zip tidak corrupt")
        raise
    
    # Cek hasil extract
    print(f"\n3️⃣ Verifikasi folder hasil extract:")
    dataset_path = dataset_extract_path
    
    if os.path.exists(dataset_path):
        print(f"   ✓ Folder {DATASET_NAME}/ ada di {dataset_path}")
        
        # List struktur
        print(f"\n4️⃣ Struktur folder:")
        root_items = os.listdir(dataset_path)
        root_items.sort()
        
        data_config = {'nc': 0, 'names': []}
        
        for item in root_items:
            item_path = os.path.join(dataset_path, item)
            
            if os.path.isdir(item_path):
                sub_items = os.listdir(item_path)
                has_images = 'images' in sub_items
                has_labels = 'labels' in sub_items
                
                if has_images or has_labels:
                    img_count = len(os.listdir(os.path.join(item_path, 'images'))) if has_images else 0
                    lbl_count = len(os.listdir(os.path.join(item_path, 'labels'))) if has_labels else 0
                    print(f"   📁 {item:12} : {img_count:4} images, {lbl_count:4} labels")
            else:
                # Jika ada file (seperti data.yaml)
                if item.endswith('.yaml') or item.endswith('.yml'):
                    print(f"   📄 {item}")
                    
                    # Baca config dari file
                    try:
                        with open(item_path, 'r') as f:
                            file_config = yaml.safe_load(f)
                        if file_config:
                            data_config['nc'] = file_config.get('nc', 0)
                            data_config['names'] = file_config.get('names', [])
                    except:
                        pass
        
        # Buat/Update data.yaml
        print(f"\n5️⃣ Setup data.yaml:")
        
        # Cari folder yang ada (train/val/test/valid)
        # Cari folder yang ada (train/val/test/valid)
        # Cari folder yang ada (khusus cross validation)
        available_folders = [
            f for f in os.listdir(dataset_path)
            if os.path.isdir(os.path.join(dataset_path, f))
            and f in ['train', 'test']
        ]
        
        if available_folders:
            # Mapping nama folder
            # Build data.yaml config khusus cross validation
            new_config = {
                'path': dataset_path,
                'train': 'train/images',
                'test': 'test/images',
                'nc': data_config.get('nc', 0),
                'names': data_config.get('names', [])
            }
            
            # Simpan data.yaml
            data_yaml_path = os.path.join(dataset_path, 'data.yaml')
            with open(data_yaml_path, 'w') as f:
                yaml.dump(new_config, f, default_flow_style=False, sort_keys=False)
            
            print(f"   ✓ data.yaml dibuat/updated: {data_yaml_path}")
            print(f"\n6️⃣ Isi data.yaml:")
            with open(data_yaml_path, 'r') as f:
                print(f.read())
            
            print(f"\n✅ DATASET SIAP DIGUNAKAN!")
            print(f"   Path dataset : {dataset_path}")
            print(f"   Path data.yaml : {data_yaml_path}")
        else:
            print(f"   ❌ Tidak ada folder train/test di dataset")
            print(f"   Folder yang ada: {root_items}")
    else:
        print(f"   ❌ Folder {DATASET_NAME} tidak ditemukan")
        print(f"   Available folders di {EXTRACT_TO_PATH}:")
        for item in os.listdir(EXTRACT_TO_PATH)[:10]:
            print(f"      - {item}")
else:
    print(f"   ❌ File zip TIDAK ditemukan: {DATASET_ZIP_PATH}")
    print(f"\n   📝 Langkah:")
    print(f"   1. Upload .zip ke Google Drive (folder MyDrive atau Path yang Anda tentukan)")
    print(f"   2. Update path di variable DATASET_ZIP_PATH")
    print(f"   3. Run cell ini lagi")

print("\n" + "=" * 70)

📦 UNZIP & SETUP DATASET BARU

1️⃣ Cek file zip:
   ✓ File zip ditemukan: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta.zip
   📊 Ukuran: 235.13 MB

2️⃣ Unzip dataset...
   ✓ Unzip selesai ke: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data

3️⃣ Verifikasi folder hasil extract:
   ✓ Folder data_coffe_augmentasi_derta/ ada di /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data

4️⃣ Struktur folder:
   📄 data.yaml
   📁 test         :  300 images,  300 labels
   📁 train        : 2398 images, 2398 labels

5️⃣ Setup data.yaml:
   ✓ data.yaml dibuat/updated: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/data.yaml

6️⃣ Isi data.yaml:
path: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data
train: train/images
test: test/images
nc: 7
names:
- Cercospora
- Cercospora dan Phoma
- Hemileia Vastatrix
- Hemileia Vastatrix dan Cercospora
- Hemileia Vastatrix dan Cercospora dan Phoma
- Hemile

**Convert Label**

In [7]:
import os

ALL_LABEL_DIRS = [
    "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels",
    "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/test/labels",
]

for LABEL_DIR in ALL_LABEL_DIRS:

    print(f"\nProcessing: {LABEL_DIR}")

    for file in os.listdir(LABEL_DIR):

        if not file.endswith(".txt"):
            continue

        path = os.path.join(LABEL_DIR, file)

        new_lines = []

        with open(path, "r") as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            # minimal polygon
            if len(parts) < 7:
                continue

            cls = parts[0]
            coords = list(map(float, parts[1:]))

            xs = coords[0::2]
            ys = coords[1::2]

            x_min = min(xs)
            x_max = max(xs)

            y_min = min(ys)
            y_max = max(ys)

            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2

            width = x_max - x_min
            height = y_max - y_min

            new_line = f"{cls} {x_center} {y_center} {width} {height}\n"
            new_lines.append(new_line)

        with open(path, "w") as f:
            f.writelines(new_lines)

print("\n✅ Semua label berhasil diubah ke YOLO Detection")


Processing: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels

Processing: /content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/test/labels

✅ Semua label berhasil diubah ke YOLO Detection


--Train Model--

**Yolo12n**

In [9]:
from ultralytics import YOLO
import yaml
import os
import time
import shutil
from pathlib import Path
import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import KFold

try:
    # =============================
    # 1. KONFIGURASI DATASET
    # =============================
    print("\n" + "="*70)
    print("K-FOLD DATASET CONFIGURATION")
    print("="*70)

    IMAGE_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/images"
    LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels"

    MODEL = "ultralytics/cfg/models/12/yolo12n.yaml"

    EPOCHS = 300
    BATCH_SIZE = 32
    IMAGE_SIZE = 640
    LEARNING_RATE = 0.001
    OPTIMIZER = "Adam"

    NUM_FOLDS = 5
    # START_FOLD = 5

    CLASS_NAMES = [
    'Cercospora',
    'Cercospora dan Phoma',
    'Hemileia Vastatrix',
    'Hemileia Vastatrix dan Cercospora',
    'Hemileia Vastatrix dan Cercospora dan Phoma',
    'Hemileia Vastatrix dan Phoma',
    'Phoma'
    ]
    NC = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 4. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
        # if fold + 1 < START_FOLD:
        #     continue

        print("\n" + "="*70)
        print(f"🚀 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir = f"/content/fold_{fold+1}"

        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")

        val_img_dir = os.path.join(fold_dir, "valid/images")
        val_lbl_dir = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)

        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        # =============================
        # COPY TRAIN FILES
        # =============================
        for idx in train_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(train_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(train_lbl_dir, label_name)
            )

        # =============================
        # COPY VALID FILES
        # =============================
        for idx in val_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(val_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(val_lbl_dir, label_name)
            )

        # =============================
        # CREATE data.yaml
        # =============================
        data_yaml = {
            'path': fold_dir,
            'train': 'train/images',
            'val': 'valid/images',
            'nc': NC,
            'names': CLASS_NAMES
        }

        yaml_path = os.path.join(fold_dir, "data.yaml")

        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # =============================
        # LOAD MODEL
        # =============================
        # =============================
        # LOAD / RESUME MODEL
        # =============================

        model_name = Path(MODEL).stem

        resume_path = f"/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12n/{model_name}_fold_{fold+1}/weights/last.pt"

        if os.path.exists(resume_path):

            print(f"\n🔄 Resume training Fold {fold+1}")

            model = YOLO(resume_path)

            RESUME = True

        else:

            print(f"\n🆕 Training baru Fold {fold+1}")

            model = YOLO(MODEL)

            RESUME = False

        # =============================
        # HITUNG PARAMETER
        # =============================
        total_params = sum(
            p.numel() for p in model.model.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in model.model.parameters()
            if p.requires_grad
        )

        print("\n🧠 MODEL INFO")
        print("-"*70)

        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")

        # =============================
        # TRAINING
        # =============================
        start_time = time.time()

        results_train = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            optimizer=OPTIMIZER,
            lr0=LEARNING_RATE,
            project="/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12n",
            name=f"{model_name}_fold_{fold+1}",
            exist_ok=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            verbose=True,
            patience=50,
            save=True,
            # resume=RESUME
        )

        train_time = time.time() - start_time

        run_dir = Path(results_train.save_dir)

        best_model_path = run_dir / "weights/best.pt"

        print(f"\n✅ Fold {fold+1} selesai")
        print(f"Training time : {train_time/60:.2f} menit")

        # =============================
        # VALIDASI
        # =============================
        model_best = YOLO(best_model_path)

        val_results = model_best.val(
            data=yaml_path,
            split="val",
            plots=False
        )

        # =============================
        # FPS
        # =============================
        print("\n🎥 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        inference_times = []

        _ = model_best.predict(
            val_images[0],
            verbose=False
        )

        for img_path in val_images:

            start_inf = time.time()

            _ = model_best.predict(
                source=img_path,
                imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            end_inf = time.time()

            inference_times.append(end_inf - start_inf)

        avg_latency = np.mean(inference_times)

        fps = 1 / avg_latency

        # =============================
        # METRIK
        # =============================
        P = val_results.results_dict.get('metrics/precision(B)', 0)
        R = val_results.results_dict.get('metrics/recall(B)', 0)

        m50 = val_results.results_dict.get('metrics/mAP50(B)', 0)

        m5095 = val_results.results_dict.get(
            'metrics/mAP50-95(B)',
            0
        )

        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n📊 HASIL FOLD")
        print("-"*70)

        print(f"Precision : {P:.4f}")
        print(f"Recall    : {R:.4f}")
        print(f"F1-Score  : {F1:.4f}")
        print(f"mAP50     : {m50:.4f}")
        print(f"mAP50-95  : {m5095:.4f}")
        print(f"FPS       : {fps:.2f}")

        # =============================
        # SAVE CSV
        # =============================
        csv_path = run_dir / "results.csv"

        if csv_path.exists():

            df = pd.read_csv(csv_path)

            df["total_params"] = total_params
            df["trainable_params"] = trainable_params
            df["fps"] = fps

            df.to_csv(csv_path, index=False)

            print("✓ results.csv updated")

        # =============================
        # SIMPAN HASIL
        # =============================
        all_results.append([
            fold + 1,
            P,
            R,
            F1,
            m50,
            m5095,
            fps
        ])

    # =============================
    # 5. HASIL AKHIR
    # =============================
    print("\n" + "="*70)
    print("📈 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP50",
            "mAP50-95",
            "FPS"
        ]
    )

    print(result_df)

    print("\n📊 RATA-RATA HASIL")
    print("-"*70)

    print(result_df.mean(numeric_only=True))

    summary_csv = "/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12n/results.csv"

    result_df.to_csv(summary_csv, index=False)

    print(f"\n✓ Summary saved : {summary_csv}")

except Exception as e:
    print(f"\n❌ Error : {e}")

    import traceback
    traceback.print_exc()


K-FOLD DATASET CONFIGURATION
Device : cuda
Total images : 2398

🚀 TRAINING FOLD 1/5

🆕 Training baru Fold 1

🧠 MODEL INFO
----------------------------------------------------------------------
Total params      : 2,602,288
Trainable params  : 2,602,272
New https://pypi.org/project/ultralytics/8.4.55 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

**Yolo12s**

In [10]:
from ultralytics import YOLO
import yaml
import os
import time
import shutil
from pathlib import Path
import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import KFold

try:
    # =============================
    # 1. KONFIGURASI DATASET
    # =============================
    print("\n" + "="*70)
    print("K-FOLD DATASET CONFIGURATION")
    print("="*70)

    IMAGE_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/images"
    LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels"

    MODEL = "ultralytics/cfg/models/12/yolo12s.yaml"

    EPOCHS = 300
    BATCH_SIZE = 32
    IMAGE_SIZE = 640
    LEARNING_RATE = 0.001
    OPTIMIZER = "Adam"

    NUM_FOLDS = 5
    # START_FOLD = 5

    CLASS_NAMES = [
    'Cercospora',
    'Cercospora dan Phoma',
    'Hemileia Vastatrix',
    'Hemileia Vastatrix dan Cercospora',
    'Hemileia Vastatrix dan Cercospora dan Phoma',
    'Hemileia Vastatrix dan Phoma',
    'Phoma'
    ]
    NC = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 4. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
        # if fold + 1 < START_FOLD:
        #     continue

        print("\n" + "="*70)
        print(f"🚀 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir = f"/content/fold_{fold+1}"

        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")

        val_img_dir = os.path.join(fold_dir, "valid/images")
        val_lbl_dir = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)

        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        # =============================
        # COPY TRAIN FILES
        # =============================
        for idx in train_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(train_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(train_lbl_dir, label_name)
            )

        # =============================
        # COPY VALID FILES
        # =============================
        for idx in val_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(val_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(val_lbl_dir, label_name)
            )

        # =============================
        # CREATE data.yaml
        # =============================
        data_yaml = {
            'path': fold_dir,
            'train': 'train/images',
            'val': 'valid/images',
            'nc': NC,
            'names': CLASS_NAMES
        }

        yaml_path = os.path.join(fold_dir, "data.yaml")

        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # =============================
        # LOAD MODEL
        # =============================
        # =============================
        # LOAD / RESUME MODEL
        # =============================

        model_name = Path(MODEL).stem

        resume_path = f"/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12s/{model_name}_fold_{fold+1}/weights/last.pt"

        if os.path.exists(resume_path):

            print(f"\n🔄 Resume training Fold {fold+1}")

            model = YOLO(resume_path)

            RESUME = True

        else:

            print(f"\n🆕 Training baru Fold {fold+1}")

            model = YOLO(MODEL)

            RESUME = False

        # =============================
        # HITUNG PARAMETER
        # =============================
        total_params = sum(
            p.numel() for p in model.model.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in model.model.parameters()
            if p.requires_grad
        )

        print("\n🧠 MODEL INFO")
        print("-"*70)

        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")

        # =============================
        # TRAINING
        # =============================
        start_time = time.time()

        results_train = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            optimizer=OPTIMIZER,
            lr0=LEARNING_RATE,
            project="/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12s",
            name=f"{model_name}_fold_{fold+1}",
            exist_ok=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            verbose=True,
            patience=50,
            save=True,
            # resume=RESUME
        )

        train_time = time.time() - start_time

        run_dir = Path(results_train.save_dir)

        best_model_path = run_dir / "weights/best.pt"

        print(f"\n✅ Fold {fold+1} selesai")
        print(f"Training time : {train_time/60:.2f} menit")

        # =============================
        # VALIDASI
        # =============================
        model_best = YOLO(best_model_path)

        val_results = model_best.val(
            data=yaml_path,
            split="val",
            plots=False
        )

        # =============================
        # FPS
        # =============================
        print("\n🎥 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        inference_times = []

        _ = model_best.predict(
            val_images[0],
            verbose=False
        )

        for img_path in val_images:

            start_inf = time.time()

            _ = model_best.predict(
                source=img_path,
                imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            end_inf = time.time()

            inference_times.append(end_inf - start_inf)

        avg_latency = np.mean(inference_times)

        fps = 1 / avg_latency

        # =============================
        # METRIK
        # =============================
        P = val_results.results_dict.get('metrics/precision(B)', 0)
        R = val_results.results_dict.get('metrics/recall(B)', 0)

        m50 = val_results.results_dict.get('metrics/mAP50(B)', 0)

        m5095 = val_results.results_dict.get(
            'metrics/mAP50-95(B)',
            0
        )

        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n📊 HASIL FOLD")
        print("-"*70)

        print(f"Precision : {P:.4f}")
        print(f"Recall    : {R:.4f}")
        print(f"F1-Score  : {F1:.4f}")
        print(f"mAP50     : {m50:.4f}")
        print(f"mAP50-95  : {m5095:.4f}")
        print(f"FPS       : {fps:.2f}")

        # =============================
        # SAVE CSV
        # =============================
        csv_path = run_dir / "results.csv"

        if csv_path.exists():

            df = pd.read_csv(csv_path)

            df["total_params"] = total_params
            df["trainable_params"] = trainable_params
            df["fps"] = fps

            df.to_csv(csv_path, index=False)

            print("✓ results.csv updated")

        # =============================
        # SIMPAN HASIL
        # =============================
        all_results.append([
            fold + 1,
            P,
            R,
            F1,
            m50,
            m5095,
            fps
        ])

    # =============================
    # 5. HASIL AKHIR
    # =============================
    print("\n" + "="*70)
    print("📈 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP50",
            "mAP50-95",
            "FPS"
        ]
    )

    print(result_df)

    print("\n📊 RATA-RATA HASIL")
    print("-"*70)

    print(result_df.mean(numeric_only=True))

    summary_csv = "/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12s/results.csv"

    result_df.to_csv(summary_csv, index=False)

    print(f"\n✓ Summary saved : {summary_csv}")

except Exception as e:
    print(f"\n❌ Error : {e}")

    import traceback
    traceback.print_exc()


K-FOLD DATASET CONFIGURATION
Device : cuda
Total images : 2398

🚀 TRAINING FOLD 1/5

🆕 Training baru Fold 1

🧠 MODEL INFO
----------------------------------------------------------------------
Total params      : 9,284,096
Trainable params  : 9,284,080
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

**Yolo12m**

In [6]:
from ultralytics import YOLO
import yaml
import os
import time
import shutil
from pathlib import Path
import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import KFold

try:
    # =============================
    # 1. KONFIGURASI DATASET
    # =============================
    print("\n" + "="*70)
    print("K-FOLD DATASET CONFIGURATION")
    print("="*70)

    IMAGE_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/images"
    LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels"

    MODEL = "ultralytics/cfg/models/12/yolo12m.yaml"

    EPOCHS = 300
    BATCH_SIZE = 32
    IMAGE_SIZE = 640
    LEARNING_RATE = 0.001
    OPTIMIZER = "Adam"

    NUM_FOLDS = 5
    START_FOLD = 5

    CLASS_NAMES = [
    'Cercospora',
    'Cercospora dan Phoma',
    'Hemileia Vastatrix',
    'Hemileia Vastatrix dan Cercospora',
    'Hemileia Vastatrix dan Cercospora dan Phoma',
    'Hemileia Vastatrix dan Phoma',
    'Phoma'
    ]
    NC = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 4. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
        if fold + 1 < START_FOLD:
            continue

        print("\n" + "="*70)
        print(f"🚀 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir = f"/content/fold_{fold+1}"

        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")

        val_img_dir = os.path.join(fold_dir, "valid/images")
        val_lbl_dir = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)

        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        # =============================
        # COPY TRAIN FILES
        # =============================
        for idx in train_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(train_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(train_lbl_dir, label_name)
            )

        # =============================
        # COPY VALID FILES
        # =============================
        for idx in val_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(val_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(val_lbl_dir, label_name)
            )

        # =============================
        # CREATE data.yaml
        # =============================
        data_yaml = {
            'path': fold_dir,
            'train': 'train/images',
            'val': 'valid/images',
            'nc': NC,
            'names': CLASS_NAMES
        }

        yaml_path = os.path.join(fold_dir, "data.yaml")

        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # =============================
        # LOAD MODEL
        # =============================
        # =============================
        # LOAD / RESUME MODEL
        # =============================

        model_name = Path(MODEL).stem

        resume_path = f"/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12m/{model_name}_fold_{fold+1}/weights/last.pt"

        if os.path.exists(resume_path):

            print(f"\n🔄 Resume training Fold {fold+1}")

            model = YOLO(resume_path)

            RESUME = True

        else:

            print(f"\n🆕 Training baru Fold {fold+1}")

            model = YOLO(MODEL)

            RESUME = False

        # =============================
        # HITUNG PARAMETER
        # =============================
        total_params = sum(
            p.numel() for p in model.model.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in model.model.parameters()
            if p.requires_grad
        )

        print("\n🧠 MODEL INFO")
        print("-"*70)

        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")

        # =============================
        # TRAINING
        # =============================
        start_time = time.time()

        results_train = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            optimizer=OPTIMIZER,
            lr0=LEARNING_RATE,
            project="/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12m",
            name=f"{model_name}_fold_{fold+1}",
            exist_ok=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            verbose=True,
            patience=50,
            save=True,
            resume=RESUME
        )

        train_time = time.time() - start_time

        run_dir = Path(results_train.save_dir)

        best_model_path = run_dir / "weights/best.pt"

        print(f"\n✅ Fold {fold+1} selesai")
        print(f"Training time : {train_time/60:.2f} menit")

        # =============================
        # VALIDASI
        # =============================
        model_best = YOLO(best_model_path)

        val_results = model_best.val(
            data=yaml_path,
            split="val",
            plots=False
        )

        # =============================
        # FPS
        # =============================
        print("\n🎥 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        inference_times = []

        _ = model_best.predict(
            val_images[0],
            verbose=False
        )

        for img_path in val_images:

            start_inf = time.time()

            _ = model_best.predict(
                source=img_path,
                imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            end_inf = time.time()

            inference_times.append(end_inf - start_inf)

        avg_latency = np.mean(inference_times)

        fps = 1 / avg_latency

        # =============================
        # METRIK
        # =============================
        P = val_results.results_dict.get('metrics/precision(B)', 0)
        R = val_results.results_dict.get('metrics/recall(B)', 0)

        m50 = val_results.results_dict.get('metrics/mAP50(B)', 0)

        m5095 = val_results.results_dict.get(
            'metrics/mAP50-95(B)',
            0
        )

        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n📊 HASIL FOLD")
        print("-"*70)

        print(f"Precision : {P:.4f}")
        print(f"Recall    : {R:.4f}")
        print(f"F1-Score  : {F1:.4f}")
        print(f"mAP50     : {m50:.4f}")
        print(f"mAP50-95  : {m5095:.4f}")
        print(f"FPS       : {fps:.2f}")

        # =============================
        # SAVE CSV
        # =============================
        csv_path = run_dir / "results.csv"

        if csv_path.exists():

            df = pd.read_csv(csv_path)

            df["total_params"] = total_params
            df["trainable_params"] = trainable_params
            df["fps"] = fps

            df.to_csv(csv_path, index=False)

            print("✓ results.csv updated")

        # =============================
        # SIMPAN HASIL
        # =============================
        all_results.append([
            fold + 1,
            P,
            R,
            F1,
            m50,
            m5095,
            fps
        ])

    # =============================
    # 5. HASIL AKHIR
    # =============================
    print("\n" + "="*70)
    print("📈 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP50",
            "mAP50-95",
            "FPS"
        ]
    )

    print(result_df)

    print("\n📊 RATA-RATA HASIL")
    print("-"*70)

    print(result_df.mean(numeric_only=True))

    summary_csv = "/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12m/results.csv"

    result_df.to_csv(summary_csv, index=False)

    print(f"\n✓ Summary saved : {summary_csv}")

except Exception as e:
    print(f"\n❌ Error : {e}")

    import traceback
    traceback.print_exc()


K-FOLD DATASET CONFIGURATION
Device : cuda
Total images : 2398

🚀 TRAINING FOLD 5/5

🔄 Resume training Fold 5

🧠 MODEL INFO
----------------------------------------------------------------------
Total params      : 20,142,885
Trainable params  : 0
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

**Yolo12l**

In [7]:
from ultralytics import YOLO
import yaml
import os
import time
import shutil
from pathlib import Path
import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import KFold

try:
    # =============================
    # 1. KONFIGURASI DATASET
    # =============================
    print("\n" + "="*70)
    print("K-FOLD DATASET CONFIGURATION")
    print("="*70)

    IMAGE_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/images"
    LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels"

    MODEL = "ultralytics/cfg/models/12/yolo12l.yaml"

    EPOCHS = 300
    BATCH_SIZE = 32
    IMAGE_SIZE = 640
    LEARNING_RATE = 0.001
    OPTIMIZER = "Adam"

    NUM_FOLDS = 5
    START_FOLD = 4

    CLASS_NAMES = [
    'Cercospora',
    'Cercospora dan Phoma',
    'Hemileia Vastatrix',
    'Hemileia Vastatrix dan Cercospora',
    'Hemileia Vastatrix dan Cercospora dan Phoma',
    'Hemileia Vastatrix dan Phoma',
    'Phoma'
    ]
    NC = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 4. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
        if fold + 1 < START_FOLD:
            continue

        print("\n" + "="*70)
        print(f"🚀 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir = f"/content/fold_{fold+1}"

        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")

        val_img_dir = os.path.join(fold_dir, "valid/images")
        val_lbl_dir = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)

        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        # =============================
        # COPY TRAIN FILES
        # =============================
        for idx in train_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(train_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(train_lbl_dir, label_name)
            )

        # =============================
        # COPY VALID FILES
        # =============================
        for idx in val_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(val_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(val_lbl_dir, label_name)
            )

        # =============================
        # CREATE data.yaml
        # =============================
        data_yaml = {
            'path': fold_dir,
            'train': 'train/images',
            'val': 'valid/images',
            'nc': NC,
            'names': CLASS_NAMES
        }

        yaml_path = os.path.join(fold_dir, "data.yaml")

        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # =============================
        # LOAD MODEL
        # =============================
        # =============================
        # LOAD / RESUME MODEL
        # =============================

        model_name = Path(MODEL).stem

        resume_path = f"/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12l/{model_name}_fold_{fold+1}/weights/last.pt"

        if os.path.exists(resume_path):

            print(f"\n🔄 Resume training Fold {fold+1}")

            model = YOLO(resume_path)

            RESUME = True

        else:

            print(f"\n🆕 Training baru Fold {fold+1}")

            model = YOLO(MODEL)

            RESUME = False

        # =============================
        # HITUNG PARAMETER
        # =============================
        total_params = sum(
            p.numel() for p in model.model.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in model.model.parameters()
            if p.requires_grad
        )

        print("\n🧠 MODEL INFO")
        print("-"*70)

        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")

        # =============================
        # TRAINING
        # =============================
        start_time = time.time()

        results_train = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            optimizer=OPTIMIZER,
            lr0=LEARNING_RATE,
            project="/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12l",
            name=f"{model_name}_fold_{fold+1}",
            exist_ok=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            verbose=True,
            patience=50,
            save=True,
            resume=RESUME
        )

        train_time = time.time() - start_time

        run_dir = Path(results_train.save_dir)

        best_model_path = run_dir / "weights/best.pt"

        print(f"\n✅ Fold {fold+1} selesai")
        print(f"Training time : {train_time/60:.2f} menit")

        # =============================
        # VALIDASI
        # =============================
        model_best = YOLO(best_model_path)

        val_results = model_best.val(
            data=yaml_path,
            split="val",
            plots=False
        )

        # =============================
        # FPS
        # =============================
        print("\n🎥 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        inference_times = []

        _ = model_best.predict(
            val_images[0],
            verbose=False
        )

        for img_path in val_images:

            start_inf = time.time()

            _ = model_best.predict(
                source=img_path,
                imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            end_inf = time.time()

            inference_times.append(end_inf - start_inf)

        avg_latency = np.mean(inference_times)

        fps = 1 / avg_latency

        # =============================
        # METRIK
        # =============================
        P = val_results.results_dict.get('metrics/precision(B)', 0)
        R = val_results.results_dict.get('metrics/recall(B)', 0)

        m50 = val_results.results_dict.get('metrics/mAP50(B)', 0)

        m5095 = val_results.results_dict.get(
            'metrics/mAP50-95(B)',
            0
        )

        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n📊 HASIL FOLD")
        print("-"*70)

        print(f"Precision : {P:.4f}")
        print(f"Recall    : {R:.4f}")
        print(f"F1-Score  : {F1:.4f}")
        print(f"mAP50     : {m50:.4f}")
        print(f"mAP50-95  : {m5095:.4f}")
        print(f"FPS       : {fps:.2f}")

        # =============================
        # SAVE CSV
        # =============================
        csv_path = run_dir / "results.csv"

        if csv_path.exists():

            df = pd.read_csv(csv_path)

            df["total_params"] = total_params
            df["trainable_params"] = trainable_params
            df["fps"] = fps

            df.to_csv(csv_path, index=False)

            print("✓ results.csv updated")

        # =============================
        # SIMPAN HASIL
        # =============================
        all_results.append([
            fold + 1,
            P,
            R,
            F1,
            m50,
            m5095,
            fps
        ])

    # =============================
    # 5. HASIL AKHIR
    # =============================
    print("\n" + "="*70)
    print("📈 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP50",
            "mAP50-95",
            "FPS"
        ]
    )

    print(result_df)

    print("\n📊 RATA-RATA HASIL")
    print("-"*70)

    print(result_df.mean(numeric_only=True))

    summary_csv = "/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12l/results.csv"

    result_df.to_csv(summary_csv, index=False)

    print(f"\n✓ Summary saved : {summary_csv}")

except Exception as e:
    print(f"\n❌ Error : {e}")

    import traceback
    traceback.print_exc()


K-FOLD DATASET CONFIGURATION
Device : cuda
Total images : 2398

🚀 TRAINING FOLD 4/5

🔄 Resume training Fold 4

🧠 MODEL INFO
----------------------------------------------------------------------
Total params      : 26,394,501
Trainable params  : 0
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

**YOLO12x**

In [1]:
from ultralytics import YOLO
import yaml
import os
import time
import shutil
from pathlib import Path
import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import KFold

try:
    # =============================
    # 1. KONFIGURASI DATASET
    # =============================
    print("\n" + "="*70)
    print("K-FOLD DATASET CONFIGURATION")
    print("="*70)

    IMAGE_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/images"
    LABEL_DIR = "/content/drive/MyDrive/Dataset_Terbaru/data_coffe_augmentasi_derta/data/train/labels"

    MODEL = "ultralytics/cfg/models/12/yolo12x.yaml"

    EPOCHS = 300
    BATCH_SIZE = 32
    IMAGE_SIZE = 640
    LEARNING_RATE = 0.001
    OPTIMIZER = "Adam"

    NUM_FOLDS = 5
    START_FOLD = 5

    CLASS_NAMES = [
    'Cercospora',
    'Cercospora dan Phoma',
    'Hemileia Vastatrix',
    'Hemileia Vastatrix dan Cercospora',
    'Hemileia Vastatrix dan Cercospora dan Phoma',
    'Hemileia Vastatrix dan Phoma',
    'Phoma'
    ]
    NC = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 4. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
        if fold + 1 < START_FOLD:
            continue

        print("\n" + "="*70)
        print(f"🚀 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir = f"/content/fold_{fold+1}"

        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")

        val_img_dir = os.path.join(fold_dir, "valid/images")
        val_lbl_dir = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)

        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        # =============================
        # COPY TRAIN FILES
        # =============================
        for idx in train_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(train_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(train_lbl_dir, label_name)
            )

        # =============================
        # COPY VALID FILES
        # =============================
        for idx in val_idx:

            img_name = image_files[idx]

            shutil.copy(
                os.path.join(IMAGE_DIR, img_name),
                os.path.join(val_img_dir, img_name)
            )

            label_name = Path(img_name).stem + ".txt"

            shutil.copy(
                os.path.join(LABEL_DIR, label_name),
                os.path.join(val_lbl_dir, label_name)
            )

        # =============================
        # CREATE data.yaml
        # =============================
        data_yaml = {
            'path': fold_dir,
            'train': 'train/images',
            'val': 'valid/images',
            'nc': NC,
            'names': CLASS_NAMES
        }

        yaml_path = os.path.join(fold_dir, "data.yaml")

        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # =============================
        # LOAD MODEL
        # =============================
        # =============================
        # LOAD / RESUME MODEL
        # =============================

        model_name = Path(MODEL).stem

        resume_path = f"/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12x/{model_name}_fold_{fold+1}/weights/last.pt"

        if os.path.exists(resume_path):

            print(f"\n🔄 Resume training Fold {fold+1}")

            model = YOLO(resume_path)

            RESUME = True

        else:

            print(f"\n🆕 Training baru Fold {fold+1}")

            model = YOLO(MODEL)

            RESUME = False

        # =============================
        # HITUNG PARAMETER
        # =============================
        total_params = sum(
            p.numel() for p in model.model.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in model.model.parameters()
            if p.requires_grad
        )

        print("\n🧠 MODEL INFO")
        print("-"*70)

        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")

        # =============================
        # TRAINING
        # =============================
        start_time = time.time()

        results_train = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            optimizer=OPTIMIZER,
            lr0=LEARNING_RATE,
            project="/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12x",
            name=f"{model_name}_fold_{fold+1}",
            exist_ok=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            verbose=True,
            patience=50,
            save=True,
            resume=RESUME
        )

        train_time = time.time() - start_time

        run_dir = Path(results_train.save_dir)

        best_model_path = run_dir / "weights/best.pt"

        print(f"\n✅ Fold {fold+1} selesai")
        print(f"Training time : {train_time/60:.2f} menit")

        # =============================
        # VALIDASI
        # =============================
        model_best = YOLO(best_model_path)

        val_results = model_best.val(
            data=yaml_path,
            split="val",
            plots=False
        )

        # =============================
        # FPS
        # =============================
        print("\n🎥 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        inference_times = []

        _ = model_best.predict(
            val_images[0],
            verbose=False
        )

        for img_path in val_images:

            start_inf = time.time()

            _ = model_best.predict(
                source=img_path,
                imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            end_inf = time.time()

            inference_times.append(end_inf - start_inf)

        avg_latency = np.mean(inference_times)

        fps = 1 / avg_latency

        # =============================
        # METRIK
        # =============================
        P = val_results.results_dict.get('metrics/precision(B)', 0)
        R = val_results.results_dict.get('metrics/recall(B)', 0)

        m50 = val_results.results_dict.get('metrics/mAP50(B)', 0)

        m5095 = val_results.results_dict.get(
            'metrics/mAP50-95(B)',
            0
        )

        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n📊 HASIL FOLD")
        print("-"*70)

        print(f"Precision : {P:.4f}")
        print(f"Recall    : {R:.4f}")
        print(f"F1-Score  : {F1:.4f}")
        print(f"mAP50     : {m50:.4f}")
        print(f"mAP50-95  : {m5095:.4f}")
        print(f"FPS       : {fps:.2f}")

        # =============================
        # SAVE CSV
        # =============================
        csv_path = run_dir / "results.csv"

        if csv_path.exists():

            df = pd.read_csv(csv_path)

            df["total_params"] = total_params
            df["trainable_params"] = trainable_params
            df["fps"] = fps

            df.to_csv(csv_path, index=False)

            print("✓ results.csv updated")

        # =============================
        # SIMPAN HASIL
        # =============================
        all_results.append([
            fold + 1,
            P,
            R,
            F1,
            m50,
            m5095,
            fps
        ])

    # =============================
    # 5. HASIL AKHIR
    # =============================
    print("\n" + "="*70)
    print("📈 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP50",
            "mAP50-95",
            "FPS"
        ]
    )

    print(result_df)

    print("\n📊 RATA-RATA HASIL")
    print("-"*70)

    print(result_df.mean(numeric_only=True))

    summary_csv = "/content/drive/MyDrive/train_yolo_data_augmentasi/Yolo12x/results.csv"

    result_df.to_csv(summary_csv, index=False)

    print(f"\n✓ Summary saved : {summary_csv}")

except Exception as e:
    print(f"\n❌ Error : {e}")

    import traceback
    traceback.print_exc()


K-FOLD DATASET CONFIGURATION
Device : cuda
Total images : 2398

🚀 TRAINING FOLD 5/5

🔄 Resume training Fold 5

🧠 MODEL INFO
----------------------------------------------------------------------
Total params      : 59,126,469
Trainable params  : 0
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez